# Análisis: preguntas de negocio y arquetipos de clientes

Este notebook usa las tablas limpias generadas por 01_limpieza.ipynb.

## 1. Carga de datos limpios

In [1]:
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent
PROC = RAIZ / "data" / "processed"
FIGS = RAIZ / "reports" / "figures"
DATA = RAIZ / "data"

archivos = ["customers", "orders", "order_items", "products"]
faltantes = [a for a in archivos if not (PROC / f"{a}.csv").exists()]
assert not faltantes, f"Faltan archivos en {PROC}: {faltantes}. Ejecuta primero 01_limpieza.ipynb"

customers   = pd.read_csv(PROC / "customers.csv",
                          parse_dates=["fecha_registro", "fecha_nacimiento", "primera_compra", "cliente_desde"])
orders      = pd.read_csv(PROC / "orders.csv", parse_dates=["fecha"])
order_items = pd.read_csv(PROC / "order_items.csv")
products    = pd.read_csv(PROC / "products.csv", parse_dates=["fecha_lanzamiento"])
stores = pd.read_csv(DATA / "stores.csv")  

FECHA_CORTE = pd.Timestamp("2026-09-30")

for nombre, df in {"customers": customers, "orders": orders, "order_items": order_items,
                   "products": products, "stores": stores}.items():
    print(f"{nombre}: {df.shape[0]} filas")

customers: 3005 filas
orders: 22651 filas
order_items: 47641 filas
products: 240 filas
stores: 5 filas


## 2. Tabla base de ventas

Se unen las líneas de venta con su orden, su producto y su cliente, y se calcula la venta neta de cada línea. Se aplican las definiciones de la limpieza:
- Solo cuentan como venta las órdenes completadas.
- Venta neta = cantidad × precio_unitario × (1 − descuento_pct / 100).
- El total de ventas de la marca incluye todas las ventas completadas; el análisis por cliente usa solo las órdenes atribuibles a un cliente registrado, sin cuentas internas.

In [3]:
ventas = (order_items
    .merge(orders, on="order_id", how="inner")
    .merge(products[["sku", "coleccion", "categoria", "fecha_lanzamiento"]], on="sku", how="left")
    .merge(customers[["customer_id", "cuenta_interna"]], on="customer_id", how="left")
    .merge(stores.rename(columns={"nombre": "tienda"}), on="tienda_id", how="left"))

ventas = ventas[ventas["es_venta"]].copy()
ventas["venta_neta"] = ventas["cantidad"] * ventas["precio_unitario"] * (1 - ventas["descuento_pct"] / 100)
ventas["cuenta_interna"] = ventas["cuenta_interna"].fillna(False).astype(bool)

# Ventas que se pueden asignar a un cliente real
ventas_cli = ventas[ventas["cliente_atribuible"] & ~ventas["cuenta_interna"]]

total_marca = ventas["venta_neta"].sum()
total_cli = ventas_cli["venta_neta"].sum()

print(f"Ventas netas de la marca: {total_marca:,.0f} MXN ({ventas['order_id'].nunique():,} órdenes)")
print(f"Ventas atribuibles a clientes: {total_cli:,.0f} MXN ({total_cli / total_marca:.1%})")
print(f"Clientes con al menos una compra: {ventas_cli['customer_id'].nunique():,}")

Ventas netas de la marca: 45,033,143 MXN (21,297 órdenes)
Ventas atribuibles a clientes: 42,247,804 MXN (93.8%)
Clientes con al menos una compra: 2,877
